# Extract Image

In [12]:
import cv2
from projectaria_tools.core import data_provider
from projectaria_tools.core.stream_id import StreamId

vrsfile = "example"
provider = data_provider.create_vrs_data_provider(f"./{vrsfile}.vrs")

stream_mappings = {
    "camera-rgb": StreamId("214-1"),
    "camera-eyetracking": StreamId("211-1"),
    #"audio": StreamId("231-1")
}

for stream_name, stream_id in stream_mappings.items():
    num_frames = provider.get_num_data(stream_id)
    first_frame = provider.get_image_data_by_index(stream_id, 0)[0].to_numpy_array()
    
    # Rotate 90° clockwise to determine new dimensions
    rotated_first_frame = cv2.rotate(first_frame, cv2.ROTATE_90_CLOCKWISE)
    height, width = rotated_first_frame.shape[:2]

    out = cv2.VideoWriter(
        f"{vrsfile}_{stream_name}.mp4",
        cv2.VideoWriter_fourcc(*"mp4v"),
        30,  # FPS
        (width, height)
    )

    for idx in range(num_frames):
        image_data = provider.get_image_data_by_index(stream_id, idx)[0].to_numpy_array()
        image_bgr = cv2.cvtColor(image_data, cv2.COLOR_RGB2BGR)
        # Rotate 90° clockwise
        image_rotated = cv2.rotate(image_bgr, cv2.ROTATE_90_CLOCKWISE)
        print(image_rotated.shape)
        out.write(image_rotated)

    out.release()
    print(f"{stream_name}.mp4 saved with {num_frames} rotated frames.")

[VRSIndexRecord][WARNING]: 2 record(s) not sorted properly. Sorting index.
[MultiRecordFileReader][DEBUG]: Opened file './example.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 211-1/camera-et activated
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][INFO]: streamId 231-1/mic activated
[VrsDataProvider][INFO]: streamId 281-1/gps activated
[VrsDataProvider][INFO]: streamId 282-1/wps activated
[VrsDataProvider][INFO]: streamId 283-1/bluetooth activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Fail to activate streamId 286-1


(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1408, 3)
(1408, 1

# Extract Audio

In [10]:
import numpy as np
import soundfile as sf
from projectaria_tools.core import data_provider

vrsfile = "example"
provider = data_provider.create_vrs_data_provider(f"./{vrsfile}.vrs")

stream_id = provider.get_stream_id_from_label("mic")
num_audio_blocks = provider.get_num_data(stream_id)

print("Number of audio blocks:", num_audio_blocks)

all_channels = None

for index in range(num_audio_blocks):

    audio_record, audio_config = provider.get_audio_data_by_index(stream_id, index)

    # ✅ Force correct dtype
    audio_block = np.array(audio_record.data, dtype=np.int32)

    # Detect channel count
    if len(audio_block) % 7 == 0:
        num_channels = 7
    elif len(audio_block) % 2 == 0:
        num_channels = 2
    else:
        raise RuntimeError("Cannot determine number of channels")

    if all_channels is None:
        all_channels = [[] for _ in range(num_channels)]
        print("Detected channels:", num_channels)

    # De-interleave
    for c in range(num_channels):
        all_channels[c].extend(audio_block[c::num_channels])

# Stack
audio_array = np.stack(
    [np.array(ch, dtype=np.int32) for ch in all_channels],
    axis=1
)

print("Final audio shape:", audio_array.shape)

# Save WAV
sf.write("output.wav", audio_array, samplerate=48000)

print("Audio saved to output.wav")

[VRSIndexRecord][WARNING]: 2 record(s) not sorted properly. Sorting index.
[MultiRecordFileReader][DEBUG]: Opened file './example.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 211-1/camera-et activated
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][INFO]: streamId 231-1/mic activated
[VrsDataProvider][INFO]: streamId 281-1/gps activated
[VrsDataProvider][INFO]: streamId 282-1/wps activated
[VrsDataProvider][INFO]: streamId 283-1/bluetooth activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Fail to activate streamId 286-1


Number of audio blocks: 379
Detected channels: 7
Final audio shape: (776192, 7)
Audio saved to output.wav


# Extract Gaze 

In [13]:
import cv2
import numpy as np
import torch

from projectaria_tools.core import data_provider
from projectaria_tools.core.stream_id import StreamId

from eye_gaze import EyeGazeInferenceBatch, AriaGazeProjector


# =============================================
# CONFIG
# =============================================
vrsfile = "example"
provider = data_provider.create_vrs_data_provider(f"./{vrsfile}.vrs")

rgb_stream_id = StreamId("214-1")
eye_stream_id = StreamId("211-1")

num_rgb_frames = provider.get_num_data(rgb_stream_id)
num_eye_frames = provider.get_num_data(eye_stream_id)

print("RGB frames:", num_rgb_frames)
print("Eye frames:", num_eye_frames)

# =============================================
# Load Gaze Model
# =============================================
print("[Gaze] Loading model...")
gaze_model = EyeGazeInferenceBatch(
    device="cuda" if torch.cuda.is_available() else "cpu"
)

gaze_projector = AriaGazeProjector(
    vrs_path=f"./{vrsfile}.vrs"
)

# =============================================
# Prepare Video Writer
# =============================================
first_rgb = provider.get_image_data_by_index(rgb_stream_id, 0)[0].to_numpy_array()
first_rgb = cv2.cvtColor(first_rgb, cv2.COLOR_RGB2BGR)
first_rgb = cv2.rotate(first_rgb, cv2.ROTATE_90_CLOCKWISE)

height, width = first_rgb.shape[:2]

out = cv2.VideoWriter(
    f"{vrsfile}_rgb_with_gaze.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    30,
    (width, height)
)

# =============================================
# Main Loop
# =============================================
num_frames = min(num_rgb_frames, num_eye_frames)

for idx in range(num_frames):

    # -------- RGB --------
    rgb = provider.get_image_data_by_index(rgb_stream_id, idx)[0].to_numpy_array()
    rgb_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    rgb_rot = cv2.rotate(rgb_bgr, cv2.ROTATE_90_CLOCKWISE)

    # -------- Eye --------
    eye_img = provider.get_image_data_by_index(eye_stream_id, idx)[0].to_numpy_array()

    # Add batch dimension
    eye_batch = eye_img[None, ...]

    # 1️⃣ Inference
    results = gaze_model.predict(eye_batch)

    # 2️⃣ Projection
    unscaled_coords, _ = gaze_projector.project_with_simple_fallback(
        yaw_array=results["yaw"],
        pitch_array=results["pitch"],
        image_width=width,
        image_height=height,
        depth_m=1.0,
    )

    gaze_x, gaze_y = unscaled_coords[0]

    # -------- Draw Gaze --------
    gaze_x = int(gaze_x)
    gaze_y = int(gaze_y)

    cv2.circle(rgb_rot, (gaze_x, gaze_y), 10, (0, 0, 255), -1)

    out.write(rgb_rot)

    if idx % 50 == 0:
        print(f"Processed frame {idx}/{num_frames}")

out.release()

print("Saved:", f"{vrsfile}_rgb_with_gaze.mp4")


[VRSIndexRecord][WARNING]: 2 record(s) not sorted properly. Sorting index.
[MultiRecordFileReader][DEBUG]: Opened file './example.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 211-1/camera-et activated
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][INFO]: streamId 231-1/mic activated
[VrsDataProvider][INFO]: streamId 281-1/gps activated
[VrsDataProvider][INFO]: streamId 282-1/wps activated
[VrsDataProvider][INFO]: streamId 283-1/bluetooth activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Fail to activate streamId 286-1
[VRSIndexRecord][WARNING]: 2 record(s) not sorted properly. Sorting index.
[MultiRecordFileReader][DEBUG]: Opened file './example.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 211-1/camera-et activated
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][INFO]: streamId 231-1/mic activated
[VrsDataProvider][INFO]: streamId

RGB frames: 489
Eye frames: 489
[Gaze] Loading model...
✅ Eye tracking model loaded on cpu
[INFO] Loading calibration from VRS: ./example.vrs
[INFO] ✓ Device calibration loaded successfully
[INFO] Using accurate projection with device calibration
Processed frame 0/489
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device ca

# full vrs execution

In [2]:
import os
import glob
import cv2
import numpy as np
import torch
import soundfile as sf

from projectaria_tools.core import data_provider
from projectaria_tools.core.stream_id import StreamId

from eye_gaze import EyeGazeInferenceBatch, AriaGazeProjector


# =============================================
# GLOBAL CONFIG
# =============================================
RGB_STREAM = StreamId("214-1")
EYE_STREAM = StreamId("211-1")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("[INFO] Loading gaze model once...")
gaze_model = EyeGazeInferenceBatch(device=DEVICE)


# =============================================

def rotate_coordinate(coordinate, image_width, image_height, rotation='cw_90'):
    """
    Rotate coordinate by various angles.
    Coordinate system: (0,0) at upper-left
    """
    x, y = coordinate
    if rotation == 'cw_90':
        # 시계방향 90도
        x_new = image_height - 1 - y
        y_new = x
    
    elif rotation == 'cw_180':
        # 180도
        x_new = image_width - 1 - x
        y_new = image_height - 1 - y
    
    elif rotation == 'cw_270' or rotation == 'ccw_90':
        # 시계방향 270도 = 반시계방향 90도
        x_new = y
        y_new = image_width - 1 - x
    
    else:
        # 회전 없음
        x_new = x
        y_new = y
    
    return np.array([x_new, y_new])

def scale_coordinate(coordinate, scale_factor=256/1408):
    """
    Scale coordinate by a factor.
    """
    x, y = coordinate
    x_new = x * scale_factor
    y_new = y * scale_factor
    return np.array([x_new, y_new])

# =============================================

# =============================================
# PROCESS ONE VRS FILE
# =============================================
def process_vrs(vrs_path):

    basename = os.path.splitext(os.path.basename(vrs_path))[0]
    print(f"\n==============================")
    print(f"[PROCESSING] {basename}")
    print("==============================")

    provider = data_provider.create_vrs_data_provider(vrs_path)

    num_rgb_frames = provider.get_num_data(RGB_STREAM)
    num_eye_frames = provider.get_num_data(EYE_STREAM)
    num_frames = min(num_rgb_frames, num_eye_frames)

    print("RGB frames:", num_rgb_frames)
    print("Eye frames:", num_eye_frames)

    gaze_projector = AriaGazeProjector(vrs_path=vrs_path)

    # ---- First frame to get size ----
    first_rgb = provider.get_image_data_by_index(RGB_STREAM, 0)[0].to_numpy_array()
    first_rgb = cv2.cvtColor(first_rgb, cv2.COLOR_RGB2BGR)
    first_rgb = cv2.rotate(first_rgb, cv2.ROTATE_90_CLOCKWISE)

    height, width = first_rgb.shape[:2]

    # ---- Video Writers ----
    rgb_writer = cv2.VideoWriter(
        f"./preproc_files/{basename}_rgb.mp4",
        cv2.VideoWriter_fourcc(*"mp4v"),
        15,
        (512, 512)
    )

    overlay_writer = cv2.VideoWriter(
        f"./preproc_files/{basename}_rgb_with_gaze.mp4",
        cv2.VideoWriter_fourcc(*"mp4v"),
        15,
        (512, 512)
    )

    gaze_coords = []
    pitch_yaws = []



    # =============================================
    # FRAME LOOP (15fps = every 2nd frame)
    # =============================================
    for idx in range(0, num_frames, 2):

        # -------- RGB --------
        rgb = provider.get_image_data_by_index(RGB_STREAM, idx)[0].to_numpy_array()
        rgb_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        rgb_rot = cv2.rotate(rgb_bgr, cv2.ROTATE_90_CLOCKWISE)
        rgb_rot_resized = cv2.resize(rgb_rot, (512, 512))

        rgb_writer.write(rgb_rot_resized)

        # -------- Eye --------
        eye_img = provider.get_image_data_by_index(EYE_STREAM, idx)[0].to_numpy_array()
        eye_batch = eye_img[None, ...]

        # Inference
        results = gaze_model.predict(eye_batch)
        pitch_yaws.append(np.array([results['pitch'][0], results['pitch_lower'][0], results['pitch_upper'][0], results['yaw'][0], results['yaw_lower'][0], results['yaw_upper'][0]]))

        # Projection
        unscaled_coords, _ = gaze_projector.project_with_simple_fallback(
            yaw_array=results["yaw"],
            pitch_array=results["pitch"],
            image_width=width,
            image_height=height,
            depth_m=1.0,
        )

        gaze_x, gaze_y = unscaled_coords[0]

        # Draw overlay
        gx = int(gaze_x)
        gy = int(gaze_y)

        coordinate = rotate_coordinate(np.array([gx,gy]), width, height, rotation='cw_90')
        coordinate = scale_coordinate(coordinate, scale_factor=512/width)

        gaze_coords.append([coordinate[0],coordinate[1]])
        
        overlay_frame = cv2.resize(rgb_rot, (512, 512))
        cv2.circle(overlay_frame, (int(coordinate[0]), int(coordinate[1])), 4, (0, 0, 255), -1)

        overlay_writer.write(overlay_frame)

        if idx % 100 == 0:
            print(f"Frame {idx}/{num_frames}")

    rgb_writer.release()
    overlay_writer.release()

    # Save gaze npz
    gaze_coords = np.array(gaze_coords)
    pitch_yaws = np.array(pitch_yaws)
    np.savez(f"./preproc_files/{basename}_gaze.npz", gaze=gaze_coords)
    np.savez(f"./preproc_files/{basename}_pitch_yaw.npz", pitch_yaw=pitch_yaws)
    print("[Saved] gaze npz")

    # =============================================
    # AUDIO
    # =============================================
    try:
        stream_id = provider.get_stream_id_from_label("mic")
        num_audio_blocks = provider.get_num_data(stream_id)

        all_channels = None

        for index in range(num_audio_blocks):

            audio_record, audio_config = provider.get_audio_data_by_index(stream_id, index)
            audio_block = np.array(audio_record.data, dtype=np.int32)

            # Auto detect channels
            if len(audio_block) % 7 == 0:
                num_channels = 7
            elif len(audio_block) % 2 == 0:
                num_channels = 2
            else:
                raise RuntimeError("Cannot determine number of channels")

            if all_channels is None:
                all_channels = [[] for _ in range(num_channels)]
                print("Detected audio channels:", num_channels)

            for c in range(num_channels):
                all_channels[c].extend(audio_block[c::num_channels])

        audio_array = np.stack(
            [np.array(ch, dtype=np.int32) for ch in all_channels],
            axis=1
        )

        sf.write(f"./preproc_files/{basename}_audio.wav", audio_array, samplerate=48000)
        print("[Saved] audio wav")

    except Exception as e:
        print("[WARNING] Audio extraction failed:", e)


# =============================================
# MAIN: PROCESS ALL VRS FILES
# =============================================
vrs_files = glob.glob("./vrs_files_260222/*.vrs")

if len(vrs_files) == 0:
    print("No .vrs files found.")
else:
    print("Found", len(vrs_files), "vrs files")

from tqdm import tqdm
for vrs in tqdm(vrs_files):
    process_vrs(vrs)

print("\nAll files processed.")

[INFO] Loading gaze model once...
✅ Eye tracking model loaded on cpu
Found 15 vrs files


  0%|          | 0/15 [00:00<?, ?it/s]


[PROCESSING] 260222_instruction_data
RGB frames: 422
Eye frames: 422
[INFO] Loading calibration from VRS: ./vrs_files_260222/260222_instruction_data.vrs
[INFO] ✓ Device calibration loaded successfully
[INFO] Using accurate projection with device calibration
Frame 0/422


[ProgressLogger][INFO]: 2026-02-22 23:44:40: Opening ./vrs_files_260222/260222_instruction_data.vrs...
[VRSIndexRecord][WARNING]: 1 record(s) not sorted properly. Sorting index.
[MultiRecordFileReader][DEBUG]: Opened file './vrs_files_260222/260222_instruction_data.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 211-1/camera-et activated
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][INFO]: streamId 231-1/mic activated
[VrsDataProvider][INFO]: streamId 281-1/gps activated
[VrsDataProvider][INFO]: streamId 282-1/wps activated
[VrsDataProvider][INFO]: streamId 283-1/bluetooth activated
[VrsDataProvider][INFO]: Utc stream found: 285-1
[VrsDataProvider][INFO]: Fail to activate streamId 286-1
[ProgressLogger][INFO]: 2026-02-22 23:44:40: Opening ./vrs_files_260222/260222_instruction_data.vrs...
[VRSIndexRecord][WARNING]: 1 record(s) not sorted properly. Sorting index.
[MultiRecordFileReader][DEBUG]: Opened file './vrs_files_260222/260222_instr

[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projectio

...

[Saved] audio wav

[PROCESSING] 260222_instruction_data_11
RGB frames: 210
Eye frames: 210
[INFO] Loading calibration from VRS: ./vrs_files_260222/260222_instruction_data_11.vrs
[INFO] ✓ Device calibration loaded successfully
[INFO] Using accurate projection with device calibration
Frame 0/210
[INFO] Using accurate projection with device calibration


[ProgressLogger][INFO]: 2026-02-22 23:46:37: Opening ./vrs_files_260222/260222_instruction_data_11.vrs...
[VRSIndexRecord][WARNING]: 1 record(s) not sorted properly. Sorting index.
[MultiRecordFileReader][DEBUG]: Opened file './vrs_files_260222/260222_instruction_data_11.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 211-1/camera-et activated
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][INFO]: streamId 231-1/mic activated
[VrsDataProvider][INFO]: streamId 281-1/gps activated
[VrsDataProvider][INFO]: streamId 282-1/wps activated
[VrsDataProvider][INFO]: streamId 283-1/bluetooth activated
[VrsDataProvider][INFO]: Utc stream found: 285-1
[VrsDataProvider][INFO]: Fail to activate streamId 286-1
[ProgressLogger][INFO]: 2026-02-22 23:46:37: Opening ./vrs_files_260222/260222_instruction_data_11.vrs...
[VRSIndexRecord][WARNING]: 1 record(s) not sorted properly. Sorting index.
[MultiRecordFileReader][DEBUG]: Opened file './vrs_files_260222/260

[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projection with device calibration
[INFO] Using accurate projectio

100%|██████████| 15/15 [02:03<00:00,  8.24s/it]

[Saved] audio wav

All files processed.


In [2]:
import numpy as np
data = np.load('/Users/dongdori/Desktop/260211_aria_vrs/preproc_files/hand_same1_gaze.npz')

In [5]:
data = np.load('/Users/dongdori/Desktop/260211_aria_vrs/preproc_files/hand_same3_pitch_yaw.npz')

In [8]:
data['pitch_yaw'].shape

(73, 6)

In [4]:
data['gaze']

array([[141.81818182, 139.45454545],
       [140.72727273, 138.36363636],
       [140.36363636, 136.90909091],
       [138.72727273, 134.90909091],
       [137.63636364, 134.36363636],
       [136.72727273, 134.36363636],
       [134.54545455, 133.63636364],
       [133.09090909, 132.54545455],
       [131.09090909, 131.81818182],
       [120.        , 141.09090909],
       [118.54545455, 139.45454545],
       [119.45454545, 138.90909091],
       [120.54545455, 137.63636364],
       [122.90909091, 137.27272727],
       [125.81818182, 138.18181818],
       [129.27272727, 140.18181818],
       [125.81818182, 151.45454545],
       [120.36363636, 175.09090909],
       [117.45454545, 155.45454545],
       [122.54545455, 167.09090909],
       [114.90909091, 184.72727273],
       [109.81818182, 164.54545455],
       [108.72727273, 158.18181818],
       [114.72727273, 163.09090909],
       [114.90909091, 158.90909091],
       [116.90909091, 159.81818182],
       [117.45454545, 164.72727273],
 